# gyaradax.quasilinear — QL calibration

**Saturation rule** (`gyaradax.quasilinear.ql_flux`):

$$
Q_{QL} \;=\; C_n \sum_{k_x, k_y>0} \frac{\Gamma_{lin}(k_x,k_y)}{|\phi_{lin}(k_x,k_y)|^2} \cdot \frac{\gamma(k_y)}{\langle k_\perp^2 \rangle(k_y)}
$$

γ-linear amplitude. Quasilinear weight `Γ_lin/|φ|²` is invariant under per-ky normalization. Geometric floor on ⟨k⊥²⟩.

**Calibration**: `fit_cn_parametric` — affine C_n (TGLF SAT1/2 style):

$$
C_n(\hat{s}, q, R/L_T, R/L_n) \;=\; a + b\,\hat{s} + c\,q + d\,R/L_T + e\,R/L_n
$$

5 coefficients, linear least squares. No log/exp pipeline.

**Pipeline** (`linear_from_fds`): load FDS, compute φ (+ apar/bpar in EM) via `_compute_fields`, per-(kx,ky) flux via `calculate_fluxes(reduce=False)` (kinetic-aware). γ from `growth.dat`. ~2 s/sim.

**Two splits available below — pick one (other commented out):**
1. **legacy split** (`raw/`): hand-picked iteration indices from the original 300-sim dataset.
2. **new_data split**: batches 1–5, 7–10 train; batch_6 + legacy ID/OOD as held-out test.

In [ ]:
import os
import sys
import re

ROOT = "/system/user/galletti/git/gyaradax"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel

jax.config.update("jax_enable_x64", True)

from gyaradax.quasilinear import (
    linear_from_fds,
    linear_run,
    ql_flux,
    ql_flux_diagnostics,
    FEATURE_NAMES,
    fit_cn,
    fit_cn_parametric,
    fit_cn_polynomial,
    fit_cn_ridge_polynomial,
    fit_cn_gbm,
    fit_cn_gp_log,
    harvest,
)

RAW = "/restricteddata/ukaea/gyrokinetics/raw"
NEW = "/restricteddata/ukaea/gyrokinetics/raw/new_data"

# ──── linear-input source ────────────────────────────────────────────────
# "fds":    load GKW's FDS file directly (~1-2 s/sim). γ from growth.dat.
# "linear": run gyaradax linearly from gk_init (~30 s/sim). γ measured by gyaradax.
LOADER_KIND = "fds"
LINEAR_N_STEPS = 1000


def _linear_loader(d):
    return linear_run(d, n_steps=LINEAR_N_STEPS, warm_start_from_fds=False)


loader = {"fds": linear_from_fds, "linear": _linear_loader}[LOADER_KIND]
print(f"loader: {LOADER_KIND}")


## 1. Build splits — pick one

Each `split` is a list of `(lin_dir, nl_dir, name)` triples. The downstream code is split-agnostic.

In [ ]:
# helpers
def expand(ranges):
    out = []
    for a, b in ranges:
        out.extend(range(a, b + 1))
    return out


def triples_from_idx(root, indices):
    return [(f"{root}/iteration_{n}_Lin", f"{root}/iteration_{n}", n) for n in indices]


def pairs_in_batch(batch_dir):
    """list of (lin_dir, nl_dir, idx) inside a single new_data/batch_N directory."""
    if not os.path.isdir(batch_dir):
        return []
    entries = os.listdir(batch_dir)
    lin = {}
    nl = set()
    pat = re.compile(r"^iteration_(\d+)(_Lin)?$")
    for e in entries:
        m = pat.match(e)
        if not m:
            continue
        if m.group(2):
            lin[m.group(1)] = e
        else:
            nl.add(m.group(1))
    out = []
    for k in sorted(nl, key=int):
        if k in lin:
            out.append(
                (os.path.join(batch_dir, lin[k]), os.path.join(batch_dir, f"iteration_{k}"), int(k))
            )
    return out


# # ──── SPLIT A: legacy split (~300 sims from raw/) ─────────────────────────
# TRAIN_TRIPLES = triples_from_idx(
#     RAW,
#     expand([(0,7),(9,12),(14,64),(67,99),(101,114),(116,130),
#             (132,147),(149,199),(201,234),(236,261),(263,299)]),
# )
# VAL_TRIPLES   = triples_from_idx(RAW, [13])
# ID_TRIPLES    = triples_from_idx(RAW, [8, 115, 131, 148, 235, 262])
# OOD_TRIPLES   = triples_from_idx(f'{RAW}/ood', [0, 1, 2, 3, 4])
# B6_TRIPLES    = []

# ──── SPLIT B: new_data — train on batches {1..5, 7..10}, hold batch_6 ────
TRAIN_BATCHES = [
    "batch_1",
    "batch_2",
    "batch_3",
    "batch_4",
    "batch_5",
    "batch_7",
    "batch_8",
    "batch_9",
    "batch_10",
]
TRAIN_TRIPLES = sum((pairs_in_batch(f"{NEW}/{b}") for b in TRAIN_BATCHES), [])
VAL_TRIPLES = triples_from_idx(RAW, [13])
ID_TRIPLES = triples_from_idx(RAW, [8, 115, 131, 148, 235, 262])
OOD_TRIPLES = triples_from_idx(f"{RAW}/ood", [0, 1, 2, 3, 4])
B6_TRIPLES = pairs_in_batch(f"{NEW}/batch_6")

print(
    f"TRAIN {len(TRAIN_TRIPLES)}  VAL {len(VAL_TRIPLES)}  "
    f"ID {len(ID_TRIPLES)}  OOD {len(OOD_TRIPLES)}  B6 {len(B6_TRIPLES)}"
)

## 2. Harvest (X, Y, F) for each split

In [ ]:
def _X_of(arr):
    return float(
        ql_flux(
            arr["growth_rate"],
            arr["phi2"],
            arr["phi2_kxy"],
            arr["eflux_kxy"],
            arr["krho"],
            arr["kxrh"],
            arr["little_g"],
            arr["ds"],
        )
    )


X_tr, Y_tr, F_tr, names_tr, sk_tr = harvest(
    TRAIN_TRIPLES, _X_of, loader=loader, max_workers=8, label="train"
)
X_va, Y_va, F_va, names_va, sk_va = harvest(
    VAL_TRIPLES, _X_of, loader=loader, max_workers=4, label="val"
)
X_id, Y_id, F_id, names_id, sk_id = harvest(
    ID_TRIPLES, _X_of, loader=loader, max_workers=4, label="ID"
)
X_oo, Y_oo, F_oo, names_oo, sk_oo = harvest(
    OOD_TRIPLES, _X_of, loader=loader, max_workers=4, label="OOD"
)
if B6_TRIPLES:
    X_b6, Y_b6, F_b6, names_b6, sk_b6 = harvest(
        B6_TRIPLES, _X_of, loader=loader, max_workers=8, label="batch6"
    )
else:
    X_b6, Y_b6, F_b6, names_b6, sk_b6 = (
        np.array([]),
        np.array([]),
        np.empty((0, len(FEATURE_NAMES))),
        [],
        0,
    )
print(
    f"train {len(X_tr)} (sk {sk_tr}) val {len(X_va)} "
    f"ID {len(X_id)} OOD {len(X_oo)} B6 {len(X_b6)}"
)

## 3. Fit QL (scalar + parametric) and baseline regressors

In [ ]:
m = (X_tr > 0) & (Y_tr > 0)
cn_scalar = float(fit_cn(jnp.asarray(X_tr[m]), jnp.asarray(Y_tr[m])))
cn_param = fit_cn_parametric(X_tr[m], Y_tr[m], F_tr[m])
print(f"scalar C_n = {cn_scalar:.4f}")
print(f"parametric: {cn_param!r}")

# polynomial C_n: strict generalization of parametric. tunable degree.
# default off. when on, fits Y ≈ X · poly_d(features) with all monomials
# of total degree ≤ POLY_DEGREE in DEFAULT_PARAM_FEATURES. d=1 == parametric.
USE_POLY_CN = False
POLY_DEGREE = 2

cn_poly = None
if USE_POLY_CN:
    cn_poly = fit_cn_polynomial(X_tr[m], Y_tr[m], F_tr[m], degree=POLY_DEGREE)
    print(f"polynomial (d={POLY_DEGREE}): {cn_poly!r}")
else:
    print(f"polynomial: disabled  (set USE_POLY_CN=True to enable)")

# advanced fitters (calibration_advanced.py) — modern alternatives to the
# closed-form OLS calibrators. All share the .predict(X, F) interface.
#   * ridge-poly  Y ≈ X · poly_d(F) with cross-validated L2 (deg tunable)
#   * gbm        log(Y/X) = HistGradientBoostingRegressor(F)
#   * gp         log(Y/X) = GP(F) with ARD Matern + WhiteKernel
USE_RIDGE_CN = True
RIDGE_DEGREE = 2
USE_GBM_CN   = True
USE_GP_CN    = True

cn_ridge = None
cn_gbm = None
cn_gp = None
if USE_RIDGE_CN:
    cn_ridge = fit_cn_ridge_polynomial(X_tr[m], Y_tr[m], F_tr[m], degree=RIDGE_DEGREE)
    print(f"ridge-poly (d={RIDGE_DEGREE}): {cn_ridge!r}")
if USE_GBM_CN:
    cn_gbm = fit_cn_gbm(X_tr[m], Y_tr[m], F_tr[m])
    print(f"gbm: {cn_gbm!r}")
if USE_GP_CN:
    cn_gp = fit_cn_gp_log(X_tr[m], Y_tr[m], F_tr[m])
    print(f"gp:  {cn_gp!r}")

f_min, f_max = F_tr.min(0), F_tr.max(0)
f_rng = np.where(f_max > f_min, f_max - f_min, 1.0)
y_min, y_max = Y_tr.min(), Y_tr.max()
y_rng = y_max - y_min if y_max > y_min else 1.0


def n_F(F):
    return (F - f_min) / f_rng


def n_Y(Y):
    return (Y - y_min) / y_rng


def d_Y(Yn):
    return Yn * y_rng + y_min


keep = (F_tr.max(0) - F_tr.min(0)) > 1e-9
print(f"kept features: {[FEATURE_NAMES[i] for i in range(len(FEATURE_NAMES)) if keep[i]]}")

ols = LinearRegression().fit(n_F(F_tr)[:, keep], n_Y(Y_tr))
kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(keep.sum()), nu=2.5) + WhiteKernel(0.01)
gpr = GaussianProcessRegressor(
    kernel=kernel, n_restarts_optimizer=3, normalize_y=False, random_state=0
).fit(n_F(F_tr)[:, keep], n_Y(Y_tr))


## 4. Evaluate — RMSE on train / val / ID / OOD / batch_6

In [ ]:
def rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt) - np.asarray(yp)) ** 2)))


def report(label, Y, F, X):
    if len(Y) == 0:
        print(f"\n=== {label} (empty) ===")
        return None
    Y_ql_sc = cn_scalar * X
    Y_ql_par = cn_param.predict(X, F)
    Y_ql_poly = cn_poly.predict(X, F) if cn_poly is not None else None
    Y_ql_ridge = cn_ridge.predict(X, F) if cn_ridge is not None else None
    Y_ql_gbm = cn_gbm.predict(X, F) if cn_gbm is not None else None
    Y_ql_gp = cn_gp.predict(X, F) if cn_gp is not None else None
    Y_ols = d_Y(ols.predict(n_F(F)[:, keep]))
    Y_gpr = d_Y(gpr.predict(n_F(F)[:, keep]))
    print(f"\n=== {label}  n={len(Y)} ===")
    print(f"  Y range: {Y.min():.3f} ... {Y.max():.3f}   mean: {Y.mean():.3f}")
    print(f"  Spearman(X, Y): {spearmanr(X, Y)[0]:+.3f}")
    print(f"  RMSE  QL scalar Cn       = {rmse(Y, Y_ql_sc):8.3f}")
    print(f"  RMSE  QL parametric Cn   = {rmse(Y, Y_ql_par):8.3f}")
    if Y_ql_poly is not None:
        print(f"  RMSE  QL polynomial Cn   = {rmse(Y, Y_ql_poly):8.3f}  (deg={cn_poly.degree})")
    if Y_ql_ridge is not None:
        print(f"  RMSE  QL ridge-poly Cn   = {rmse(Y, Y_ql_ridge):8.3f}  (deg={cn_ridge.degree}, α={cn_ridge.alpha:.2e})")
    if Y_ql_gbm is not None:
        print(f"  RMSE  QL gbm Cn          = {rmse(Y, Y_ql_gbm):8.3f}")
    if Y_ql_gp is not None:
        print(f"  RMSE  QL gp Cn           = {rmse(Y, Y_ql_gp):8.3f}")
    print(f"  RMSE  OLS features       = {rmse(Y, Y_ols):8.3f}")
    print(f"  RMSE  GPR features       = {rmse(Y, Y_gpr):8.3f}")
    print(f"  RMSE  mean-train         = {rmse(Y, np.full_like(Y, Y_tr.mean())):8.3f}")
    return dict(
        Y_ql_sc=Y_ql_sc,
        Y_ql_par=Y_ql_par,
        Y_ql_poly=Y_ql_poly,
        Y_ql_ridge=Y_ql_ridge,
        Y_ql_gbm=Y_ql_gbm,
        Y_ql_gp=Y_ql_gp,
        Y_ols=Y_ols,
        Y_gpr=Y_gpr,
    )


_ = report("TRAIN", Y_tr, F_tr, X_tr)
rva = report("VAL (iter_13)", Y_va, F_va, X_va)
rid = report("TEST ID", Y_id, F_id, X_id)
roo = report("TEST OOD", Y_oo, F_oo, X_oo)
rb6 = report("TEST batch_6", Y_b6, F_b6, X_b6)


## 5. Per-sim breakdown (val + small test sets)

In [ ]:
def show(label, names, Y, F, X, res):
    if res is None:
        return
    cols = [("Y_true", lambda i: Y[i]),
            ("QL_sc",  lambda i: res["Y_ql_sc"][i]),
            ("QL_par", lambda i: res["Y_ql_par"][i])]
    if res.get("Y_ql_poly") is not None:
        cols.append(("QL_poly", lambda i: res["Y_ql_poly"][i]))
    if res.get("Y_ql_ridge") is not None:
        cols.append(("QL_ridge", lambda i: res["Y_ql_ridge"][i]))
    if res.get("Y_ql_gbm") is not None:
        cols.append(("QL_gbm", lambda i: res["Y_ql_gbm"][i]))
    if res.get("Y_ql_gp") is not None:
        cols.append(("QL_gp", lambda i: res["Y_ql_gp"][i]))
    cols.append(("OLS", lambda i: res["Y_ols"][i]))
    cols.append(("GPR", lambda i: res["Y_gpr"][i]))

    header = f"  {'name':>14s}" + "".join(f" {c[0]:>10s}" for c in cols)
    print(f"\n{label}:")
    print(header)
    for i, n in enumerate(names):
        row = f"  iter_{n:<9d}" + "".join(f" {c[1](i):>10.3f}" for c in cols)
        print(row)


show("VAL", names_va, Y_va, F_va, X_va, rva)
show("ID", names_id, Y_id, F_id, X_id, rid)
show("OOD", names_oo, Y_oo, F_oo, X_oo, roo)


## 6. Per-ky diagnostic for one sim (val)

In [ ]:
lin_va = VAL_TRIPLES[0][0]
n_va = VAL_TRIPLES[0][2]
arr = loader(lin_va)
diag = ql_flux_diagnostics(
    arr["growth_rate"],
    arr["phi2"],
    arr["phi2_kxy"],
    arr["eflux_kxy"],
    arr["krho"],
    arr["kxrh"],
    arr["little_g"],
    arr["ds"],
)
krho = np.asarray(arr["krho"])
gamma = np.asarray(arr["growth_rate"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(krho, gamma, "o-", ms=4)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_xlabel("krho")
axes[0].set_ylabel("γ")
axes[0].set_title("growth rate")
axes[0].grid(alpha=0.3)
axes[1].plot(krho, np.asarray(diag["sat_amp"]), "o-", ms=4)
axes[1].set_xlabel("krho")
axes[1].set_ylabel("γ / <k_⊥²>")
axes[1].set_title("SAT amplitude")
axes[1].grid(alpha=0.3)
wsum = np.asarray(diag["w_kxy"]).sum(axis=0)
axes[2].plot(krho, wsum, "o-", ms=4)
axes[2].set_xlabel("krho")
axes[2].set_ylabel("Σ_kx W")
axes[2].set_title("QL weight Σ_kx (eflux/|φ|²)")
axes[2].grid(alpha=0.3)
plt.suptitle(
    f'iter_{n_va}  Q_QL_sc={cn_scalar * float(diag["Q_QL"]):.3f}   Y_NL={Y_va[0]:.3f}', y=1.02
)
plt.tight_layout()
plt.show()